# CNN for MNIST Digit Classification (PyTorch)

A clean, fully-working convolutional neural network (CNN) for classifying handwritten digits (0-9) from the MNIST dataset, built with PyTorch by subclassing `nn.Module`.

This is an adapted version of **Lab 2, Part 1** from MIT's [Introduction to Deep Learning (6.S191)](http://introtodeeplearning.com) course. The original lab is a guided exercise with `#TODO` sections for students to fill in; this version has all TODOs completed and has been trimmed down to focus on the CNN model:

- Removed the fully-connected-only baseline model, Colab/copyright banners, and extra visualization/demo cells so only the core CNN pipeline remains
-[Comet ML](https://www.comet.com/) integration for experiment tracking (optional — add your own API key to use it)

**Architecture:** 2 convolutional layers (each followed by ReLU + max pooling) → flatten → 2 fully connected layers → 10-class output (digits 0–9).

Original course materials: [introtodeeplearning.com](http://introtodeeplearning.com) · [GitHub](https://github.com/MITDeepLearning/introtodeeplearning)

In [1]:
# Core PyTorch imports
import torch
import torch.nn as nn                       # building blocks for neural network layers
import torch.optim as optim                 # optimizers (SGD, Adam, etc.)
import torchvision
import torchvision.datasets as datasets      # gives us access to the MNIST dataset
import torchvision.transforms as transforms  # for converting images to tensors
from torch.utils.data import DataLoader      # batches and shuffles the dataset for training

# Used only for the optional single-image prediction check at the end
import matplotlib.pyplot as plt

In [ ]:
# Use the GPU if one is available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Comet ML (experiment tracking)

[Comet](https://www.comet.com/) logs your training/test metrics to a dashboard so you can track and compare runs. This is optional — the notebook works fine without it — but if you have a Comet account, paste your API key below to enable it.

In [ ]:
# Install and import Comet ML
!pip install comet_ml --quiet
import comet_ml

# TODO: paste your Comet API key here (find it at https://www.comet.com/api/my/settings)
COMET_API_KEY = ""
assert COMET_API_KEY != "", "Please insert your Comet API Key"

# Start a Comet experiment. Everything logged with `experiment.log_*` below
# will show up in your Comet dashboard for this project.
comet_ml.init(project_name="mnist-cnn-pytorch")
experiment = comet_ml.Experiment()

## 1. Load the MNIST dataset

In [ ]:
# Download and transform the MNIST dataset
transform = transforms.Compose([
    # Convert images to PyTorch tensors, which also scales pixel values from [0,255] to [0,1]
    transforms.ToTensor()
])

# Download training and test datasets (60,000 train / 10,000 test, 28x28 grayscale digit images)
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [6]:
# Create DataLoaders to feed the data to the model in shuffled batches
BATCH_SIZE = 64
trainset_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 2. Define the CNN model

The network is defined by subclassing `nn.Module`, the standard/recommended way to build models in PyTorch. It's composed of:

- 2 convolutional layers (`nn.Conv2d`), each followed by a ReLU activation and a max pooling layer (`nn.MaxPool2d`) that downsamples the feature maps
- A `nn.Flatten` layer to collapse the final feature maps into a 1D vector
- 2 fully connected (`nn.Linear`) layers that map the extracted features to class scores (logits) for the 10 digit classes

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # First convolutional layer: takes the 1-channel (grayscale) 28x28 image
        # and learns 24 different 3x3 filters -> outputs 24 feature maps
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=24, kernel_size=3)

        # First max pooling layer: downsamples each feature map by taking the
        # max value in every non-overlapping 2x2 window (halves height & width)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second convolutional layer: takes the 24 feature maps from conv1
        # and learns 36 new 3x3 filters -> outputs 36 feature maps
        self.conv2 = nn.Conv2d(in_channels=24, out_channels=36, kernel_size=3)

        # Second max pooling layer, same idea as pool1
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Flattens the (36, 5, 5) feature maps into a single 1D vector of length 36*5*5
        self.flatten = nn.Flatten()

        # First fully connected layer: maps the flattened features to 128 hidden units
        self.fc1 = nn.Linear(36 * 5 * 5, 128)

        # Shared ReLU activation, reused after each conv/linear layer that needs one
        self.relu = nn.ReLU()

        # Final linear layer: outputs one raw score (logit) per digit class (0-9).
        # We output raw logits (no softmax) because nn.CrossEntropyLoss applies
        # softmax internally during training.
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # First convolutional block: conv -> activation -> pooling
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)

        # Second convolutional block: conv -> activation -> pooling
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)

        # Flatten the extracted features, then classify with the fully connected layers
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)  # final logits, shape: (batch_size, 10)

        return x

# Instantiate the model and move it to the GPU/CPU
cnn_model = CNN().to(device)

# Run one dummy image through the model just to make sure the architecture works
# and all layer shapes line up correctly (also initializes lazy parameters, if any)
image, label = train_dataset[0]
image = image.to(device).unsqueeze(0)  # add a batch dimension -> shape becomes (1, 1, 28, 28)
output = cnn_model(image)

# Print the model architecture
print(cnn_model)

## 3. Set up the loss function, optimizer, and training hyperparameters

In [8]:
# Rebuild the model fresh (so training starts from random weights, not the dummy forward pass above)
cnn_model = CNN().to(device)

# Training hyperparameters
batch_size = 64
epochs = 7
optimizer = optim.SGD(cnn_model.parameters(), lr=1e-2)

# Cross entropy loss is the standard choice for multi-class classification;
# it combines a softmax + negative log-likelihood loss and expects raw logits as input
loss_function = nn.CrossEntropyLoss()

# Rebuild the DataLoaders in case batch_size was changed above
trainset_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 4. Train the model

In [9]:
def train(model, dataloader, criterion, optimizer, epochs, experiment=None):
    model.train()  # set the model to training mode (enables things like dropout/batchnorm updates)

    step = 0  # global batch counter, used as the x-axis for Comet's step-level charts

    for epoch in range(epochs):
        total_loss = 0
        correct_pred = 0
        total_pred = 0

        for images, labels in dataloader:
            # Move the batch to the same device as the model (GPU or CPU)
            images, labels = images.to(device), labels.to(device)

            # Forward pass: compute the model's predictions (logits)
            logits = model(images)

            # Compute the loss between the predictions and the true labels
            loss = criterion(logits, labels)

            # Backward pass: reset old gradients, compute new ones, then update the weights
            optimizer.zero_grad()  # clear gradients from the previous step
            loss.backward()        # compute gradients via backpropagation
            optimizer.step()       # update model parameters using those gradients

            # Log this batch's loss to Comet (if an experiment was passed in)
            if experiment is not None:
                experiment.log_metric("batch_loss", loss.item(), step=step)
            step += 1

            # Accumulate loss (weighted by batch size, since the last batch may be smaller)
            total_loss += loss.item() * images.size(0)

            # Track how many predictions were correct in this batch
            predicted = torch.argmax(logits, dim=1)  # class with the highest score
            correct_pred += (predicted == labels).sum().item()
            total_pred += labels.size(0)

        # Compute metrics for the epoch
        total_epoch_loss = total_loss / total_pred
        epoch_accuracy = correct_pred / total_pred
        print(f"Epoch {epoch + 1}, Loss: {total_epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

        # Log epoch-level metrics to Comet
        if experiment is not None:
            experiment.log_metric("epoch_loss", total_epoch_loss, step=epoch)
            experiment.log_metric("epoch_accuracy", epoch_accuracy, step=epoch)

In [ ]:
# Train the CNN (pass `experiment` so training metrics are logged to Comet)
train(cnn_model, trainset_loader, loss_function, optimizer, epochs, experiment=experiment)

## 5. Evaluate the model on the test dataset

In [ ]:
def evaluate(model, dataloader, loss_function):
    # Evaluate model performance on the test dataset
    model.eval()  # set the model to evaluation mode (disables dropout/batchnorm updates)
    test_loss = 0
    correct_pred = 0
    total_pred = 0

    # Disable gradient calculations, since we're not training here (saves memory & compute)
    with torch.no_grad():
        for images, labels in dataloader:
            # Move the batch to the same device as the model
            images, labels = images.to(device), labels.to(device)

            # Forward pass: get the model's predictions (logits)
            outputs = model(images)

            loss = loss_function(outputs, labels)

            # Accumulate the loss (weighted by batch size)
            test_loss += loss.item() * images.size(0)

            # Identify the predicted digit (highest-scoring class) for each image
            predicted = torch.argmax(outputs, dim=1)

            # Tally correct predictions and total predictions made so far
            correct_pred += (predicted == labels).sum().item()
            total_pred += labels.size(0)

    # Compute average loss and overall accuracy across the whole dataset
    test_loss /= total_pred
    test_acc = correct_pred / total_pred
    return test_loss, test_acc

# Run evaluation on the test set
test_loss, test_acc = evaluate(cnn_model, testset_loader, loss_function)
print('Test accuracy:', test_acc)

# Log final test metrics to Comet, then close out the experiment
experiment.log_metric("test_loss", test_loss)
experiment.log_metric("test_accuracy", test_acc)
experiment.end()

## 6. Make a prediction on a single test image

In [ ]:
# Grab the first image from the test set
test_image, test_label = test_dataset[0]
test_image = test_image.to(device).unsqueeze(0)  # add batch dimension -> shape (1, 1, 28, 28)

# Put the model in evaluation (inference) mode
cnn_model.eval()
with torch.no_grad():
    logits = cnn_model(test_image)

    # Convert logits to probabilities with softmax
    probabilities = torch.nn.functional.softmax(logits, dim=1)

# Identify the digit with the highest predicted probability
predicted_digit = torch.argmax(probabilities, dim=1).item()

print("Predicted digit:", predicted_digit)
print("True label:", test_label)

plt.imshow(test_image[0, 0, :, :].cpu(), cmap=plt.cm.binary)
plt.title(f"Predicted: {predicted_digit} | True: {test_label}")
plt.axis('off')
plt.show()